# ABSA Preprocessing

This notebook prepares the data for **Aspect-Based Sentiment Analysis (ABSA)**.
It cleans review text, parses the aspect annotations, and creates:

- `train_preprocessed`: one row per review
- `train_absa`: one row per `(review, aspect)` pair
- `validation_absa`: validation set in the same format


In [13]:
import re
import ast
import json
import numpy as np
import pandas as pd

from camel_tools.utils.normalize import (
    normalize_alef_ar,
    normalize_alef_maksura_ar,
    normalize_teh_marbuta_ar,
)
from camel_tools.utils.dediac import dediac_ar
from camel_tools.tokenizers.word import simple_word_tokenize

pd.set_option('display.max_colwidth', 200)

train_df = pd.read_excel('train_fixed.xlsx')
valid_df = pd.read_excel('validation_fixed.xlsx')
unlabeled_df = pd.read_excel('unlabeled_fixed.xlsx')

print('train shape    :', train_df.shape)
print('validation shape:', valid_df.shape)
print('unlabeled shape :', unlabeled_df.shape)
train_df.head()

train shape    : (1971, 9)
validation shape: (500, 9)
unlabeled shape : (7047, 7)


,review_id,review_text,star_rating,date,business_name,business_category,platform,aspects,aspect_sentiments
0,7238,لا يوجد الدفع بالبطاقه عند الاستلام,3,2026-03-08 00:00:00,Noon,ecommerce,play_store,"[""app_experience"", ""delivery""]","{""app_experience"": ""negative"", ""delivery"": ""negative""}"
1,1036,المكان نضيف وجميل وقعدته تحفه والخدمة فوق الممتاز والجو جميل مكان اكتر من رائع بصراحة ❤️❤️❤️❤️,5,قبل يومين (2),ممشي مصر Mawlana Cafe,كافيه,google_maps,"[""cleanliness"", ""ambiance"", ""service""]","{""cleanliness"": ""positive"", ""ambiance"": ""positive"", ""service"": ""positive""}"
2,1975,تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالولي نص ساعة فعد ساعة ونص\nالاكل يعني كويس ولكن كمية قليلة جدا,1,قبل شهر,بيت لحم Beet Lahm,مطعم,google_maps,"[""service"", ""delivery"", ""food""]","{""service"": ""negative"", ""delivery"": ""negative"", ""food"": ""neutral""}"
3,3024,احلي مكان فزايد,5,قبل شهر,ذا بلكون كافيه الشيخ زايد,مطعم مأكولات ومشروبات,google_maps,"[""general""]","{""general"": ""positive""}"
4,5483,الفطير حلو جدا\nالاحجام تحفة\nبالنسبه للسعر فا يعتبر من احسن الاسعار\n\nالبيتزا للاسف ما فيش اوبشن تختار عجينه سميكه ولا رفيعه كلها رفيعه\nحتس لو طلبتها سميكه هتجيلك رفيعه\nانما الحشو تمام و طعم ح...,4,قبل سنة,The Best Restaurant,مطعم,google_maps,"[""food"", ""price""]","{""food"": ""positive"", ""price"": ""positive""}"


In [14]:
valid_df.isna().sum()

review_id            0
review_text          0
star_rating          0
date                 0
business_name        0
business_category    0
platform             0
aspects              0
aspect_sentiments    0
dtype: int64

In [11]:
!pip install camel-tools

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/608.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/608.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/608.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/608.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/608.4 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/608.4 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/608.4 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/608.4 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/608.4 kB ? eta -:--:--
   ----------------- --

In [15]:
SENTIMENT_MAP = {'negative': 0, 'neutral': 1, 'positive': 2}
REVERSE_SENTIMENT_MAP = {v: k for k, v in SENTIMENT_MAP.items()}

ARABIC_RE = re.compile(r'[\u0600-\u06FF]')

def contains_arabic(text):
    return bool(ARABIC_RE.search(str(text)))

def normalize_arabic_with_camel(text):
    text = str(text)
    text = dediac_ar(text)
    text = normalize_alef_ar(text)
    text = normalize_alef_maksura_ar(text)
    text = normalize_teh_marbuta_ar(text)
    text = re.sub(r'ـ+', '', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    return text

def clean_arabic_text(text):
    text = str(text)
    text = text.replace('\n', ' ')
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+|#\w+', ' ', text)
    text = normalize_arabic_with_camel(text)
    tokens = simple_word_tokenize(text)
    text = ' '.join(tokens)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_non_arabic_text(text):
    text = str(text).lower()
    text = text.replace('\n', ' ')
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+|#\w+', ' ', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def clean_text(text):
    if pd.isna(text):
        return ''
    text = str(text)
    if contains_arabic(text):
        return clean_arabic_text(text)
    return clean_non_arabic_text(text)

def safe_parse(value, default):
    if pd.isna(value):
        return default
    if isinstance(value, (list, dict)):
        return value
    value = str(value).strip()
    if not value:
        return default
    try:
        return json.loads(value)
    except Exception:
        try:
            return ast.literal_eval(value)
        except Exception:
            return default

def parse_aspects(value):
    parsed = safe_parse(value, [])
    if isinstance(parsed, str):
        parsed = [parsed]
    if not isinstance(parsed, list):
        return []
    return [str(x).strip() for x in parsed if str(x).strip()]

def parse_aspect_sentiments(value):
    parsed = safe_parse(value, {})
    if not isinstance(parsed, dict):
        return {}
    cleaned = {}
    for aspect, sentiment in parsed.items():
        aspect = str(aspect).strip()
        sentiment = str(sentiment).strip().lower()
        if aspect and sentiment in SENTIMENT_MAP:
            cleaned[aspect] = sentiment
    return cleaned

def preprocess_reviews(df, labeled=True):
    df = df.copy()
    df['review_text'] = df['review_text'].fillna('').astype(str)
    df['clean_text'] = df['review_text'].apply(clean_text)
    df['is_arabic'] = df['review_text'].apply(contains_arabic)
    df['text_length'] = df['clean_text'].str.len()
    df['word_count'] = df['clean_text'].str.split().str.len()
    df['star_rating'] = pd.to_numeric(df['star_rating'], errors='coerce')

    if labeled:
        df['aspect_list'] = df['aspects'].apply(parse_aspects)
        df['aspect_sentiment_dict'] = df['aspect_sentiments'].apply(parse_aspect_sentiments)
        df['num_aspects'] = df['aspect_list'].apply(len)
    else:
        df['aspect_list'] = [[] for _ in range(len(df))]
        df['aspect_sentiment_dict'] = [{} for _ in range(len(df))]
        df['num_aspects'] = 0

    return df


In [17]:
train_df.columns

Index(['review_id', 'review_text', 'star_rating', 'date', 'business_name',
       'business_category', 'platform', 'aspects', 'aspect_sentiments'],
      dtype='object')

In [16]:
train_preprocessed = preprocess_reviews(train_df, labeled=True)
validation_preprocessed =preprocess_reviews(valid_df, labeled=True)
unlabeled_preprocessed = preprocess_reviews(unlabeled_df, labeled=False)

train_preprocessed[['review_text', 'clean_text', 'is_arabic', 'aspect_list', 'aspect_sentiment_dict', 'num_aspects']].head(10)

,review_text,clean_text,is_arabic,aspect_list,aspect_sentiment_dict,num_aspects
0,لا يوجد الدفع بالبطاقه عند الاستلام,لا يوجد الدفع بالبطاقه عند الاستلام,True,"[app_experience, delivery]","{'app_experience': 'negative', 'delivery': 'negative'}",2
1,المكان نضيف وجميل وقعدته تحفه والخدمة فوق الممتاز والجو جميل مكان اكتر من رائع بصراحة ❤️❤️❤️❤️,المكان نضيف وجميل وقعدته تحفه والخدمه فوق الممتاز والجو جميل مكان اكتر من رائع بصراحه ❤️ ❤️ ❤️ ❤️,True,"[cleanliness, ambiance, service]","{'cleanliness': 'positive', 'ambiance': 'positive', 'service': 'positive'}",3
2,تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالولي نص ساعة فعد ساعة ونص\nالاكل يعني كويس ولكن كمية قليلة جدا,تجربه سيئه سالتهم الاكل هياخد وقت قد ايه قالولي نص ساعه فعد ساعه ونص الاكل يعني كويس ولكن كميه قليله جدا,True,"[service, delivery, food]","{'service': 'negative', 'delivery': 'negative', 'food': 'neutral'}",3
3,احلي مكان فزايد,احلي مكان فزايد,True,[general],{'general': 'positive'},1
4,الفطير حلو جدا\nالاحجام تحفة\nبالنسبه للسعر فا يعتبر من احسن الاسعار\n\nالبيتزا للاسف ما فيش اوبشن تختار عجينه سميكه ولا رفيعه كلها رفيعه\nحتس لو طلبتها سميكه هتجيلك رفيعه\nانما الحشو تمام و طعم ح...,الفطير حلو جدا الاحجام تحفه بالنسبه للسعر فا يعتبر من احسن الاسعار البيتزا للاسف ما فيش اوبشن تختار عجينه سميكه ولا رفيعه كلها رفيعه حتس لو طلبتها سميكه هتجيلك رفيعه انما الحشو تمام و طعم حلو الكر...,True,"[food, price]","{'food': 'positive', 'price': 'positive'}",2
5,لقد حملت تطبيق عن طريق مستر بيست واريد سيارة تيسلا,لقد حملت تطبيق عن طريق مستر بيست واريد سياره تيسلا,True,[none],{'none': 'neutral'},1
6,من أجمل المطاعم التي اكلت فيها أكل عربي يمني.\nويتميز المطعم بالإتساع وتعدد الأماكن الخاصةمما يعطيك الإحساس بالراحة انت وعائلتك أو أصدقاؤك .,من اجمل المطاعم التي اكلت فيها اكل عربي يمني . ويتميز المطعم بالاتساع وتعدد الاماكن الخاصهمما يعطيك الاحساس بالراحه انت وعائلتك او اصدقاؤك .,True,"[food, ambiance]","{'food': 'positive', 'ambiance': 'positive'}",2
7,جميلة,جميله,True,[general],{'general': 'positive'},1
8,"The place is great , clean , service excellent, the food is very delicious, excellent quality\nChef Nawaf is doing his job perfectly 👌🏼\nالمكان أكثر من رائع ، النضافة عشرة على عشرة، الخدمة ممتازة ...","The place is great , clean , service excellent , the food is very delicious , excellent quality Chef Nawaf is doing his job perfectly 👌🏼 المكان اكثر من رائع ، النضافه عشره علي عشره ، الخدمه ممتازه...",True,"[ambiance, cleanliness, service, food]","{'ambiance': 'positive', 'cleanliness': 'positive', 'service': 'positive', 'food': 'positive'}",4
9,سى جدا جدا جدا\nوالله والله والله\nمفيش مسؤول موجود.\nمطعم فاشل فاشل,سي جدا جدا جدا والله والله والله مفيش مسؤول موجود . مطعم فاشل فاشل,True,[service],{'service': 'negative'},1


In [20]:
train_preprocessed.columns

Index(['review_id', 'review_text', 'star_rating', 'date', 'business_name',
       'business_category', 'platform', 'aspects', 'aspect_sentiments',
       'clean_text', 'is_arabic', 'text_length', 'word_count', 'aspect_list',
       'aspect_sentiment_dict', 'num_aspects'],
      dtype='object')

In [21]:
def explode_absa_dataset(df):
    rows = []

    for _, row in df.iterrows():
        aspects = row['aspect_list']
        sentiment_dict = row['aspect_sentiment_dict']

        for aspect in aspects:
            sentiment = sentiment_dict.get(aspect)

            if sentiment not in SENTIMENT_MAP:
                continue

            rows.append({
                'review_id': row['review_id'],
                'review_text': row['review_text'],
                'clean_text': row['clean_text'],
                'is_arabic': row.get('is_arabic', False),
                'business_name': row.get('business_name', ''),
                'business_category': row.get('business_category', ''),
                'platform': row.get('platform', ''),
                'star_rating': row.get('star_rating', np.nan),
                'aspect': aspect,
                'sentiment': sentiment,
                'sentiment_label': SENTIMENT_MAP[sentiment],
                'model_input': f"[ASPECT] {aspect} [TEXT] {row['clean_text']}"
            })

    return pd.DataFrame(rows)

train_absa = explode_absa_dataset(train_preprocessed)
validation_absa = explode_absa_dataset(validation_preprocessed)

print('train_absa shape     :', train_absa.shape)
print('validation_absa shape:', validation_absa.shape)
train_absa.head(10)

train_absa shape     : (3333, 12)
validation_absa shape: (840, 12)


,review_id,review_text,clean_text,is_arabic,business_name,business_category,platform,star_rating,aspect,sentiment,sentiment_label,model_input
0,7238,لا يوجد الدفع بالبطاقه عند الاستلام,لا يوجد الدفع بالبطاقه عند الاستلام,True,Noon,ecommerce,play_store,3,app_experience,negative,0,[ASPECT] app_experience [TEXT] لا يوجد الدفع بالبطاقه عند الاستلام
1,7238,لا يوجد الدفع بالبطاقه عند الاستلام,لا يوجد الدفع بالبطاقه عند الاستلام,True,Noon,ecommerce,play_store,3,delivery,negative,0,[ASPECT] delivery [TEXT] لا يوجد الدفع بالبطاقه عند الاستلام
2,1036,المكان نضيف وجميل وقعدته تحفه والخدمة فوق الممتاز والجو جميل مكان اكتر من رائع بصراحة ❤️❤️❤️❤️,المكان نضيف وجميل وقعدته تحفه والخدمه فوق الممتاز والجو جميل مكان اكتر من رائع بصراحه ❤️ ❤️ ❤️ ❤️,True,ممشي مصر Mawlana Cafe,كافيه,google_maps,5,cleanliness,positive,2,[ASPECT] cleanliness [TEXT] المكان نضيف وجميل وقعدته تحفه والخدمه فوق الممتاز والجو جميل مكان اكتر من رائع بصراحه ❤️ ❤️ ❤️ ❤️
3,1036,المكان نضيف وجميل وقعدته تحفه والخدمة فوق الممتاز والجو جميل مكان اكتر من رائع بصراحة ❤️❤️❤️❤️,المكان نضيف وجميل وقعدته تحفه والخدمه فوق الممتاز والجو جميل مكان اكتر من رائع بصراحه ❤️ ❤️ ❤️ ❤️,True,ممشي مصر Mawlana Cafe,كافيه,google_maps,5,ambiance,positive,2,[ASPECT] ambiance [TEXT] المكان نضيف وجميل وقعدته تحفه والخدمه فوق الممتاز والجو جميل مكان اكتر من رائع بصراحه ❤️ ❤️ ❤️ ❤️
4,1036,المكان نضيف وجميل وقعدته تحفه والخدمة فوق الممتاز والجو جميل مكان اكتر من رائع بصراحة ❤️❤️❤️❤️,المكان نضيف وجميل وقعدته تحفه والخدمه فوق الممتاز والجو جميل مكان اكتر من رائع بصراحه ❤️ ❤️ ❤️ ❤️,True,ممشي مصر Mawlana Cafe,كافيه,google_maps,5,service,positive,2,[ASPECT] service [TEXT] المكان نضيف وجميل وقعدته تحفه والخدمه فوق الممتاز والجو جميل مكان اكتر من رائع بصراحه ❤️ ❤️ ❤️ ❤️
5,1975,تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالولي نص ساعة فعد ساعة ونص\nالاكل يعني كويس ولكن كمية قليلة جدا,تجربه سيئه سالتهم الاكل هياخد وقت قد ايه قالولي نص ساعه فعد ساعه ونص الاكل يعني كويس ولكن كميه قليله جدا,True,بيت لحم Beet Lahm,مطعم,google_maps,1,service,negative,0,[ASPECT] service [TEXT] تجربه سيئه سالتهم الاكل هياخد وقت قد ايه قالولي نص ساعه فعد ساعه ونص الاكل يعني كويس ولكن كميه قليله جدا
6,1975,تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالولي نص ساعة فعد ساعة ونص\nالاكل يعني كويس ولكن كمية قليلة جدا,تجربه سيئه سالتهم الاكل هياخد وقت قد ايه قالولي نص ساعه فعد ساعه ونص الاكل يعني كويس ولكن كميه قليله جدا,True,بيت لحم Beet Lahm,مطعم,google_maps,1,delivery,negative,0,[ASPECT] delivery [TEXT] تجربه سيئه سالتهم الاكل هياخد وقت قد ايه قالولي نص ساعه فعد ساعه ونص الاكل يعني كويس ولكن كميه قليله جدا
7,1975,تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالولي نص ساعة فعد ساعة ونص\nالاكل يعني كويس ولكن كمية قليلة جدا,تجربه سيئه سالتهم الاكل هياخد وقت قد ايه قالولي نص ساعه فعد ساعه ونص الاكل يعني كويس ولكن كميه قليله جدا,True,بيت لحم Beet Lahm,مطعم,google_maps,1,food,neutral,1,[ASPECT] food [TEXT] تجربه سيئه سالتهم الاكل هياخد وقت قد ايه قالولي نص ساعه فعد ساعه ونص الاكل يعني كويس ولكن كميه قليله جدا
8,3024,احلي مكان فزايد,احلي مكان فزايد,True,ذا بلكون كافيه الشيخ زايد,مطعم مأكولات ومشروبات,google_maps,5,general,positive,2,[ASPECT] general [TEXT] احلي مكان فزايد
9,5483,الفطير حلو جدا\nالاحجام تحفة\nبالنسبه للسعر فا يعتبر من احسن الاسعار\n\nالبيتزا للاسف ما فيش اوبشن تختار عجينه سميكه ولا رفيعه كلها رفيعه\nحتس لو طلبتها سميكه هتجيلك رفيعه\nانما الحشو تمام و طعم ح...,الفطير حلو جدا الاحجام تحفه بالنسبه للسعر فا يعتبر من احسن الاسعار البيتزا للاسف ما فيش اوبشن تختار عجينه سميكه ولا رفيعه كلها رفيعه حتس لو طلبتها سميكه هتجيلك رفيعه انما الحشو تمام و طعم حلو الكر...,True,The Best Restaurant,مطعم,google_maps,4,food,positive,2,[ASPECT] food [TEXT] الفطير حلو جدا الاحجام تحفه بالنسبه للسعر فا يعتبر من احسن الاسعار البيتزا للاسف ما فيش اوبشن تختار عجينه سميكه ولا رفيعه كلها رفيعه حتس لو طلبتها سميكه هتجيلك رفيعه انما الحش...


In [22]:
print(train_absa['sentiment'].value_counts())
print(train_absa['sentiment_label'].value_counts())

print(train_absa['aspect'].value_counts().head(20))


sentiment
positive    1646
negative    1538
neutral      149
Name: count, dtype: int64
sentiment_label
2    1646
0    1538
1     149
Name: count, dtype: int64
aspect
service           988
food              454
app_experience    453
ambiance          378
price             354
general           303
cleanliness       185
delivery          161
none               57
Name: count, dtype: int64


In [23]:
train_final = train_absa[
    ['review_id', 'clean_text', 'aspect', 'sentiment', 'sentiment_label',
     'star_rating', 'business_category', 'platform', 'is_arabic', 'model_input']
].copy()

validation_final = validation_absa[
    ['review_id', 'clean_text', 'aspect', 'sentiment', 'sentiment_label',
     'star_rating', 'business_category', 'platform', 'is_arabic', 'model_input']
].copy()

train_final.head()


,review_id,clean_text,aspect,sentiment,sentiment_label,star_rating,business_category,platform,is_arabic,model_input
0,7238,لا يوجد الدفع بالبطاقه عند الاستلام,app_experience,negative,0,3,ecommerce,play_store,True,[ASPECT] app_experience [TEXT] لا يوجد الدفع بالبطاقه عند الاستلام
1,7238,لا يوجد الدفع بالبطاقه عند الاستلام,delivery,negative,0,3,ecommerce,play_store,True,[ASPECT] delivery [TEXT] لا يوجد الدفع بالبطاقه عند الاستلام
2,1036,المكان نضيف وجميل وقعدته تحفه والخدمه فوق الممتاز والجو جميل مكان اكتر من رائع بصراحه ❤️ ❤️ ❤️ ❤️,cleanliness,positive,2,5,كافيه,google_maps,True,[ASPECT] cleanliness [TEXT] المكان نضيف وجميل وقعدته تحفه والخدمه فوق الممتاز والجو جميل مكان اكتر من رائع بصراحه ❤️ ❤️ ❤️ ❤️
3,1036,المكان نضيف وجميل وقعدته تحفه والخدمه فوق الممتاز والجو جميل مكان اكتر من رائع بصراحه ❤️ ❤️ ❤️ ❤️,ambiance,positive,2,5,كافيه,google_maps,True,[ASPECT] ambiance [TEXT] المكان نضيف وجميل وقعدته تحفه والخدمه فوق الممتاز والجو جميل مكان اكتر من رائع بصراحه ❤️ ❤️ ❤️ ❤️
4,1036,المكان نضيف وجميل وقعدته تحفه والخدمه فوق الممتاز والجو جميل مكان اكتر من رائع بصراحه ❤️ ❤️ ❤️ ❤️,service,positive,2,5,كافيه,google_maps,True,[ASPECT] service [TEXT] المكان نضيف وجميل وقعدته تحفه والخدمه فوق الممتاز والجو جميل مكان اكتر من رائع بصراحه ❤️ ❤️ ❤️ ❤️


In [24]:
all_aspects = sorted(set(train_absa['aspect']).union(set(validation_absa['aspect'])))
print('number of unique aspects:', len(all_aspects))
print(all_aspects)

aspect_frequency = train_absa['aspect'].value_counts().reset_index()
aspect_frequency.columns = ['aspect', 'count']
aspect_frequency.head(20)

number of unique aspects: 9
['ambiance', 'app_experience', 'cleanliness', 'delivery', 'food', 'general', 'none', 'price', 'service']


,aspect,count
0,service,988
1,food,454
2,app_experience,453
3,ambiance,378
4,price,354
5,general,303
6,cleanliness,185
7,delivery,161
8,none,57


In [25]:
sentiment_distribution = train_absa['sentiment'].value_counts(normalize=True).mul(100).round(2)
aspect_sentiment_distribution = pd.crosstab(train_absa['aspect'], train_absa['sentiment'])

print('sentiment distribution (%)')
print(sentiment_distribution)

aspect_sentiment_distribution.head(20)

sentiment distribution (%)
sentiment
positive    49.38
negative    46.14
neutral      4.47
Name: proportion, dtype: float64


sentiment,negative,neutral,positive
aspect,,,
ambiance,101,9,268
app_experience,327,15,111
cleanliness,75,1,109
delivery,142,1,18
food,177,31,246
general,34,14,255
none,0,57,0
price,233,11,110
service,449,10,529


In [33]:
train_final[train_final["sentiment"] == "neutral"]

,review_id,clean_text,aspect,sentiment,sentiment_label,star_rating,business_category,platform,is_arabic,model_input
7,1975,تجربه سيئه سالتهم الاكل هياخد وقت قد ايه قالولي نص ساعه فعد ساعه ونص الاكل يعني كويس ولكن كميه قليله جدا,food,neutral,1,1,مطعم,google_maps,True,[ASPECT] food [TEXT] تجربه سيئه سالتهم الاكل هياخد وقت قد ايه قالولي نص ساعه فعد ساعه ونص الاكل يعني كويس ولكن كميه قليله جدا
11,8003,لقد حملت تطبيق عن طريق مستر بيست واريد سياره تيسلا,none,neutral,1,1,ecommerce,play_store,True,[ASPECT] none [TEXT] لقد حملت تطبيق عن طريق مستر بيست واريد سياره تيسلا
40,915,مطعم محترم والخدمه محترمه جدا والاكل كويس والاسعار مابقتش تفرق كتير,price,neutral,1,4,مطعم,google_maps,True,[ASPECT] price [TEXT] مطعم محترم والخدمه محترمه جدا والاكل كويس والاسعار مابقتش تفرق كتير
80,8633,الصراحه هو تطبيق حلو وكل حاجه بس لو رجع زي زمان زمان كان تقدر ترجع وتسرع برحتك في الاغنيه وكمان ممكن تحمل الاغنيه الي انت عاوزها,app_experience,neutral,1,3,entertainment,play_store,True,[ASPECT] app_experience [TEXT] الصراحه هو تطبيق حلو وكل حاجه بس لو رجع زي زمان زمان كان تقدر ترجع وتسرع برحتك في الاغنيه وكمان ممكن تحمل الاغنيه الي انت عاوزها
83,6034,البوتيك حلو يستحق الزياره بس موقعه غلط هو قدام مطعم ذا مود يارب تنتبهون,ambiance,neutral,1,3,متجر ملابس حريمي,google_maps,True,[ASPECT] ambiance [TEXT] البوتيك حلو يستحق الزياره بس موقعه غلط هو قدام مطعم ذا مود يارب تنتبهون
...,...,...,...,...,...,...,...,...,...,...
3275,8193,لا اله الاالله محمد رسول الله,none,neutral,1,5,ecommerce,play_store,True,[ASPECT] none [TEXT] لا اله الاالله محمد رسول الله
3281,3189,ليال فيين ؟ ؟,none,neutral,1,1,فندق,google_maps,True,[ASPECT] none [TEXT] ليال فيين ؟ ؟
3306,648,السلام عليكم وكل عام وانتم بخير لو سمحت لوبشتغل ممبار ممكن ابعت لك عينه تشوفها لو عجبك ابعت لك وكل عام وانتم بخير,none,neutral,1,1,مطعم عائلي,google_maps,True,[ASPECT] none [TEXT] السلام عليكم وكل عام وانتم بخير لو سمحت لوبشتغل ممبار ممكن ابعت لك عينه تشوفها لو عجبك ابعت لك وكل عام وانتم بخير
3307,8442,برنامج مفيد جدا لكن في مشكله وهي عند الرجوع خطوه للخلف يرجع للشاشه الرئيسيه يرجي المبادره بتعديلها لكم مني كل الاحترام والتقدير,app_experience,neutral,1,5,real_estate,play_store,True,[ASPECT] app_experience [TEXT] برنامج مفيد جدا لكن في مشكله وهي عند الرجوع خطوه للخلف يرجع للشاشه الرئيسيه يرجي المبادره بتعديلها لكم مني كل الاحترام والتقدير


In [ ]:
train_final[train_final["aspect"] == "none"]

,review_id,clean_text,aspect,sentiment,sentiment_label,star_rating,business_category,platform,is_arabic,model_input
11,8003,لقد حملت تطبيق عن طريق مستر بيست واريد سياره تيسلا,none,neutral,1,1,ecommerce,play_store,True,[ASPECT] none [TEXT] لقد حملت تطبيق عن طريق مستر بيست واريد سياره تيسلا
255,7673,شكران,none,neutral,1,5,ecommerce,play_store,True,[ASPECT] none [TEXT] شكران
287,8016,انا حملت هذا التطبيق من مستر بيست واتمني ان يحضرني معه في فيديو,none,neutral,1,5,ecommerce,play_store,True,[ASPECT] none [TEXT] انا حملت هذا التطبيق من مستر بيست واتمني ان يحضرني معه في فيديو
289,115,shady,none,neutral,1,5,فندق,google_maps,False,[ASPECT] none [TEXT] shady
380,4995,لوكيشن خطا,none,neutral,1,1,متجر ملابس رجالي,google_maps,True,[ASPECT] none [TEXT] لوكيشن خطا
393,8040,قرات وشاهدت تحدياتكم واريد ان اشاركم بالمسابقات 😍 لكي احصل ع الاموال,none,neutral,1,2,ecommerce,play_store,True,[ASPECT] none [TEXT] قرات وشاهدت تحدياتكم واريد ان اشاركم بالمسابقات 😍 لكي احصل ع الاموال
397,7401,الامان,none,neutral,1,5,ecommerce,play_store,True,[ASPECT] none [TEXT] الامان
406,8913,جديد,none,neutral,1,5,entertainment,play_store,True,[ASPECT] none [TEXT] جديد
474,5915,دعم,none,neutral,1,5,متجر ملابس,google_maps,True,[ASPECT] none [TEXT] دعم
480,3360,السلام عليكم كل عام وانتم بخير لو سمحت لوبشتغل ممبار ممكن ابعت لك عينه تشوفها لو عجبك نتواصل,none,neutral,1,2,مطعم أطباق اللحوم,google_maps,True,[ASPECT] none [TEXT] السلام عليكم كل عام وانتم بخير لو سمحت لوبشتغل ممبار ممكن ابعت لك عينه تشوفها لو عجبك نتواصل


In [34]:
train_preprocessed.to_csv('train_preprocessed.csv', index=False, encoding='utf-8-sig')
validation_preprocessed.to_csv('validation_preprocessed.csv', index=False, encoding='utf-8-sig')
unlabeled_preprocessed.to_csv('unlabeled_preprocessed.csv', index=False, encoding='utf-8-sig')

train_final.to_csv('train_absa_final.csv', index=False, encoding='utf-8-sig')
validation_final.to_csv('validation_absa_final.csv', index=False, encoding='utf-8-sig')

print('Preprocessed files saved successfully.')


Preprocessed files saved successfully.


In [35]:
train_final.duplicated().sum()


0

In [37]:
train_final['sentiment'].value_counts()

sentiment
positive    1646
negative    1538
neutral      149
Name: count, dtype: int64

In [42]:
train_preprocessed['word_count'] = train_preprocessed['clean_text'].fillna('').str.split().str.len()
validation_preprocessed['word_count'] = validation_preprocessed['clean_text'].fillna('').str.split().str.len()
unlabeled_preprocessed['word_count'] = unlabeled_preprocessed['clean_text'].fillna('').str.split().str.len()

train_absa = train_absa.merge(
    train_preprocessed[['review_id', 'word_count']],
    on='review_id',
    how='left'
)

validation_absa = validation_absa.merge(
    validation_preprocessed[['review_id', 'word_count']],
    on='review_id',
    how='left'
)

train_final = train_final.merge(
    train_preprocessed[['review_id', 'word_count']],
    on='review_id',
    how='left'
)

validation_final = validation_final.merge(
    validation_preprocessed[['review_id', 'word_count']],
    on='review_id',
    how='left'
)


In [43]:
train_final[train_final['word_count'] <= 1].head(10)


,review_id,clean_text,aspect,sentiment,sentiment_label,star_rating,business_category,platform,is_arabic,model_input,word_count
14,9273,جميله,general,positive,2,5,entertainment,play_store,True,[ASPECT] general [TEXT] جميله,1
26,5580,💙💙,general,positive,2,5,عِيادة أسنان,google_maps,False,[ASPECT] general [TEXT] 💙💙,1
49,7386,حلو,general,positive,2,5,ecommerce,play_store,True,[ASPECT] general [TEXT] حلو,1
98,2840,ممتاز,general,positive,2,5,فندق,google_maps,True,[ASPECT] general [TEXT] ممتاز,1
222,5559,ممتازه,general,positive,2,5,صيدلية,google_maps,True,[ASPECT] general [TEXT] ممتازه,1
233,5341,nice,general,positive,2,5,مطعم مأكولات سورية,google_maps,False,[ASPECT] general [TEXT] nice,1
245,9005,بفلوس,price,negative,0,3,entertainment,play_store,True,[ASPECT] price [TEXT] بفلوس,1
255,7673,شكران,none,neutral,1,5,ecommerce,play_store,True,[ASPECT] none [TEXT] شكران,1
289,115,shady,none,neutral,1,5,فندق,google_maps,False,[ASPECT] none [TEXT] shady,1
318,7279,تمام,general,positive,2,5,ecommerce,play_store,True,[ASPECT] general [TEXT] تمام,1
